> **Public release note.** Notebook outputs have been removed because the underlying Malaysian Motor claims data are confidential. Execution counts have also been cleared to provide clean public versions of the notebooks. Local user-specific paths and individual claim identifiers have been removed. The repository documents the data-processing, modelling and validation workflow used in the dissertation, but the numerical results cannot be reproduced end-to-end without the confidential input data.


# 05.9A — Historical Valuation-Date Diagonal Deployment (Memory-Safe)

**Purpose:** deploy the corrected Random Forest and XGBoost
future-development models at one historical valuation date and compare them
with an incurred Chain Ladder projection on the same information cut-off.

This notebook uses:

- Random Forest as the principal claim-level model;
- XGBoost as the challenger;
- the same clean feature manifest and fitted model objects;
- one historical valuation date \(T_0\);
- the latest available observation date \(T_1\);
- accident-year and portfolio comparisons.

The fixed DEV_QTR models are deployed in maturity bands:

- DEV_QTR 4--7: DEV_QTR_4 model;
- DEV_QTR 8--11: DEV_QTR_8 model;
- DEV_QTR 12--15: DEV_QTR_12 model;
- DEV_QTR above 15: full incurred position at \(T_0\), with no ML tail in
  this first deployment;
- DEV_QTR below 4: excluded from the main comparison because no early
  maturity model has been fitted.

The notebook reports the size of the excluded and mature components so that
these assumptions remain visible.

**Memory-safe revision:** the master parquet is streamed in row batches
and the fitted models are loaded one maturity band at a time. This avoids
loading the full 12-million-row feature table and all six fitted models
into memory simultaneously.


This cell sets up the first historical valuation-date back-test. It specifies where to find the cleaned snapshot inputs and fitted RF/XGBoost models, creates a folder for the back-test results, and sets the main back-testing rules before running any calculations.

There are three key settings: the historical comparison looks four quarters ahead, the current accident year is excluded since it may not have enough maturity for a matching ML model, and only Chain Ladder development factors with at least five contributing origin quarters are used.

In [ ]:
from pathlib import Path
import json
import warnings

import gc
import joblib
import numpy as np
import pandas as pd

PROJECT_FOLDER = Path(
    "/path/to/BI_large_claims_project"
)

CLEAN_INPUT_FOLDER = (
    PROJECT_FOLDER
    / "processed"
    / "chapter5_outputs"
    / "section_5_4A_clean_snapshot_model_inputs"
)

RF_MODEL_FOLDER = (
    PROJECT_FOLDER
    / "processed"
    / "chapter5_outputs"
    / "section_5_7A_clean_rf_future_development"
)

XGB_MODEL_FOLDER = (
    PROJECT_FOLDER
    / "processed"
    / "chapter5_outputs"
    / "section_5_8A_clean_xgboost_future_development"
)

OUTPUT_FOLDER = (
    PROJECT_FOLDER
    / "processed"
    / "chapter5_outputs"
    / "section_5_9A_historical_valuation_diagonal"
)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# Initial historical comparison horizon.
# Change this only before the notebook is run and documented.
BACKTEST_HORIZON_QTRS = 4

# Exclude the current accident year at T0 because some cohorts have
# less than four quarters of maturity and no matching ML model.
EXCLUDE_CURRENT_ACCIDENT_YEAR = True

# Minimum number of contributing origin quarters required to use a CL factor.
MIN_ORIGINS_PER_CL_FACTOR = 5

print("Project folder exists:", PROJECT_FOLDER.exists())
print("Output folder:", OUTPUT_FOLDER)

## Load the clean feature definition and fitted models

This section loads the saved feature definition from the earlier cleaning notebook and matches each fitted model to the right development-quarter band for the historical back-test. For example, the DEV4 model is used for claims in development quarters 4 to 7, DEV8 for 8 to 11, and DEV12 for 12 to 15.

It also checks that the needed Random Forest and XGBoost model files are present before starting the back-test. It connects each claim maturity band to the right trained model and confirms that all required files are ready.

In [ ]:
with open(CLEAN_INPUT_FOLDER / "feature_manifest.json") as f:
    manifest = json.load(f)

FEATURE_COLUMNS = manifest["feature_columns"]
CATEGORICAL_FEATURES = manifest["categorical_features"]
NUMERIC_FEATURES = manifest["numeric_features"]

SNAPSHOT_MODELS = {
    "DEV_QTR_4": {"lower": 4, "upper": 7},
    "DEV_QTR_8": {"lower": 8, "upper": 11},
    "DEV_QTR_12": {"lower": 12, "upper": 15},
}

RF_MODEL_PATHS = {}
XGB_MODEL_PATHS = {}

for snapshot in SNAPSHOT_MODELS:
    rf_path = (
        RF_MODEL_FOLDER
        / snapshot
        / f"rf_future_development_{snapshot}.joblib"
    )
    xgb_path = (
        XGB_MODEL_FOLDER
        / snapshot
        / f"xgb_future_development_{snapshot}.joblib"
    )

    if not rf_path.exists():
        raise FileNotFoundError(f"Missing RF model: {rf_path}")
    if not xgb_path.exists():
        raise FileNotFoundError(f"Missing XGBoost model: {xgb_path}")

    RF_MODEL_PATHS[snapshot] = rf_path
    XGB_MODEL_PATHS[snapshot] = xgb_path

print("Model files confirmed for:", list(SNAPSHOT_MODELS))
print("Models will be loaded one maturity band at a time to limit memory use.")

## Read the minimum date and financial fields

Quarter indices use the same convention as the existing modelling files:

\[
	ext{ACCIDENT\_QTR\_INDEX}=4	imes	ext{ACC\_YEAR}+	ext{ACC\_QTR},
\]

and:

\[
	ext{VALUATION\_QTR\_INDEX}
=
	ext{ACCIDENT\_QTR\_INDEX}+	ext{DEV\_QTR}.
\]

This section loads only the key timing and financial fields needed from the master claims-history file, checks that the required columns are present, and converts them into consistent numeric formats. It then creates an accident-quarter index and a valuation-quarter index so each claim record can be placed on an exact historical valuation timeline.

In [ ]:
key_candidates = ["SOURCE_FILE", "CLAIMS_KEY"]
timing_columns = ["ACC_YEAR", "ACC_QTR", "DEV_QTR"]
financial_columns = ["CUM_INC_LARGE", "CLAIMS_CNT"]

date_columns = list(dict.fromkeys(
    key_candidates + timing_columns + financial_columns
))

date_df = pd.read_parquet(MASTER_PATH, columns=date_columns)

required_date_columns = set(key_candidates + timing_columns + ["CUM_INC_LARGE"])
missing_date_columns = sorted(required_date_columns.difference(date_df.columns))
if missing_date_columns:
    raise ValueError(
        f"Master file is missing required fields: {missing_date_columns}"
    )

date_df["ACC_YEAR"] = pd.to_numeric(date_df["ACC_YEAR"], errors="raise").astype(int)
date_df["ACC_QTR"] = pd.to_numeric(date_df["ACC_QTR"], errors="raise").astype(int)
date_df["DEV_QTR"] = pd.to_numeric(date_df["DEV_QTR"], errors="raise").astype(int)
date_df["CUM_INC_LARGE"] = pd.to_numeric(
    date_df["CUM_INC_LARGE"], errors="coerce"
).fillna(0.0)

date_df["ACCIDENT_QTR_INDEX"] = (
    4 * date_df["ACC_YEAR"] + date_df["ACC_QTR"]
)
date_df["VALUATION_QTR_INDEX"] = (
    date_df["ACCIDENT_QTR_INDEX"] + date_df["DEV_QTR"]
)

if "CLAIMS_CNT" in date_df.columns:
    date_df["CLAIMS_CNT"] = pd.to_numeric(
        date_df["CLAIMS_CNT"], errors="coerce"
    ).fillna(0.0)

print("Rows:", f"{len(date_df):,}")
print(
    "Valuation-index range:",
    int(date_df["VALUATION_QTR_INDEX"].min()),
    "to",
    int(date_df["VALUATION_QTR_INDEX"].max()),
)

This section sets the two dates for the historical back-test. It uses the latest valuation quarter in the master data as T1T_1, then goes back four quarters to define T0T_0. Both quarter indices are then converted into readable labels like “2021 Q4”

In [ ]:
def qtr_index_to_label(index_value):
    index_value = int(index_value)
    year = (index_value - 1) // 4
    quarter = index_value - 4 * year
    return f"{year} Q{quarter}"


T1_INDEX = int(date_df["VALUATION_QTR_INDEX"].max())
T0_INDEX = T1_INDEX - BACKTEST_HORIZON_QTRS

T1_LABEL = qtr_index_to_label(T1_INDEX)
T0_LABEL = qtr_index_to_label(T0_INDEX)

print("Historical valuation date T0:", T0_LABEL, f"({T0_INDEX})")
print("Later observation date T1:", T1_LABEL, f"({T1_INDEX})")
print("Back-test horizon:", BACKTEST_HORIZON_QTRS, "quarters")

## Candidate-date and maturity audit

The table below shows how the portfolio would be divided for several
possible horizons. The selected horizon remains the value set in
`BACKTEST_HORIZON_QTRS`.

This cell is part of the audit process. It shows how much of the portfolio falls into each modeling maturity band for different historical valuation dates, before the final back-test horizon is set.

In [ ]:
def latest_claim_position_as_at(df, valuation_index):
    eligible = df.loc[
        df["VALUATION_QTR_INDEX"] <= valuation_index
    ].copy()

    eligible = eligible.sort_values(
        ["SOURCE_FILE", "CLAIMS_KEY", "VALUATION_QTR_INDEX", "DEV_QTR"]
    )

    latest = eligible.drop_duplicates(
        subset=["SOURCE_FILE", "CLAIMS_KEY"],
        keep="last",
    ).copy()

    return latest


candidate_audit_rows = []

for horizon in [4, 6, 8]:
    candidate_t0 = T1_INDEX - horizon
    candidate_snapshot = latest_claim_position_as_at(date_df, candidate_t0)

    candidate_snapshot["maturity_group"] = np.select(
        [
            candidate_snapshot["DEV_QTR"].between(0, 3),
            candidate_snapshot["DEV_QTR"].between(4, 7),
            candidate_snapshot["DEV_QTR"].between(8, 11),
            candidate_snapshot["DEV_QTR"].between(12, 15),
            candidate_snapshot["DEV_QTR"] > 15,
        ],
        [
            "UNDER_4",
            "DEV_QTR_4_BAND",
            "DEV_QTR_8_BAND",
            "DEV_QTR_12_BAND",
            "OVER_15",
        ],
        default="UNCLASSIFIED",
    )

    for group, group_df in candidate_snapshot.groupby("maturity_group"):
        candidate_audit_rows.append({
            "horizon_quarters": horizon,
            "T0_index": candidate_t0,
            "T0_label": qtr_index_to_label(candidate_t0),
            "T1_label": T1_LABEL,
            "maturity_group": group,
            "claim_count": len(group_df),
            "incurred_bixs_at_T0": float(group_df["CUM_INC_LARGE"].sum()),
        })

candidate_audit = pd.DataFrame(candidate_audit_rows)
display(candidate_audit)

candidate_audit.to_csv(
    OUTPUT_FOLDER / "historical_candidate_date_maturity_audit.csv",
    index=False,
)

## Build the claim-level position at \(T_0\) and outcome at \(T_1\)

This cell creates the historical valuation-date portfolio that the models will be tested on and records how each claim should be handled before the RF, XGBoost and Chain Ladder comparisons are made.

In [ ]:
t0_position = latest_claim_position_as_at(date_df, T0_INDEX)
t1_position = latest_claim_position_as_at(date_df, T1_INDEX)

t1_outcome = t1_position[
    ["SOURCE_FILE", "CLAIMS_KEY", "CUM_INC_LARGE"]
].rename(
    columns={"CUM_INC_LARGE": "actual_latest_bixs_T1"}
)

t0_position = t0_position.merge(
    t1_outcome,
    on=["SOURCE_FILE", "CLAIMS_KEY"],
    how="left",
    validate="one_to_one",
)

t0_position["actual_latest_bixs_T1"] = (
    t0_position["actual_latest_bixs_T1"].fillna(
        t0_position["CUM_INC_LARGE"]
    )
)
t0_position["actual_future_development_T0_to_T1"] = (
    t0_position["actual_latest_bixs_T1"]
    - t0_position["CUM_INC_LARGE"]
)

t0_year = (T0_INDEX - 1) // 4

if EXCLUDE_CURRENT_ACCIDENT_YEAR:
    eligible_year_max = t0_year - 1
    t0_position["included_in_main_diagonal"] = (
        t0_position["ACC_YEAR"] <= eligible_year_max
    )
else:
    eligible_year_max = int(t0_position["ACC_YEAR"].max())
    t0_position["included_in_main_diagonal"] = True

t0_position["model_band"] = np.select(
    [
        t0_position["DEV_QTR"].between(4, 7),
        t0_position["DEV_QTR"].between(8, 11),
        t0_position["DEV_QTR"].between(12, 15),
        t0_position["DEV_QTR"] > 15,
        t0_position["DEV_QTR"] < 4,
    ],
    [
        "DEV_QTR_4",
        "DEV_QTR_8",
        "DEV_QTR_12",
        "MATURE_POSITION",
        "UNSUPPORTED_EARLY",
    ],
    default="UNCLASSIFIED",
)

deployment_audit = (
    t0_position
    .groupby(
        ["included_in_main_diagonal", "model_band"],
        as_index=False,
    )
    .agg(
        claim_count=("CLAIMS_KEY", "count"),
        snapshot_bixs_T0=("CUM_INC_LARGE", "sum"),
        actual_latest_bixs_T1=("actual_latest_bixs_T1", "sum"),
        actual_future_development=(
            "actual_future_development_T0_to_T1", "sum"
        ),
    )
)

display(deployment_audit)
deployment_audit.to_csv(
    OUTPUT_FOLDER / "historical_diagonal_deployment_audit.csv",
    index=False,
)

print("Main diagonal accident years: through", eligible_year_max)

## Release unneeded claim-key history before loading modelling features

The full date audit contains repeated claim identifiers for all development
quarters. After the \(T_0\) and \(T_1\) claim positions have been constructed,
only a compact aggregate-development table is needed for Chain Ladder.
Releasing the larger object reduces peak memory use.

this cell reduces the dataset to the essential triangle fields and clears unnecessary data from memory so the remaining back-test calculations are more efficient.

In [ ]:
triangle_date_df = date_df[
    [
        "ACCIDENT_QTR_INDEX",
        "ACC_YEAR",
        "DEV_QTR",
        "CUM_INC_LARGE",
        "VALUATION_QTR_INDEX",
    ]
].copy()

del date_df
gc.collect()

print(
    "Retained compact triangle rows:",
    f"{len(triangle_date_df):,}",
)

## Load full snapshot features only for the selected \(T_0\) rows

The master file is read again with the locked modelling features. The
selected rows are merged using claim identity and DEV_QTR.

In [ ]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(MASTER_PATH)
available_columns = set(parquet_file.schema_arrow.names)

required_feature_source_columns = set(
    FEATURE_COLUMNS
    + [
        "SOURCE_FILE",
        "CLAIMS_KEY",
        "ACC_YEAR",
        "ACC_QTR",
        "DEV_QTR",
        "CUM_INC_LARGE",
    ]
)

# This flag is reconstructed from the snapshot amount.
required_feature_source_columns.discard("is_bi_excess_at_snapshot")

missing_feature_source_columns = sorted(
    required_feature_source_columns.difference(available_columns)
)
if missing_feature_source_columns:
    raise ValueError(
        "The master file does not contain the following modelling fields: "
        f"{missing_feature_source_columns}"
    )

feature_source_columns = sorted(required_feature_source_columns)
key_columns = ["SOURCE_FILE", "CLAIMS_KEY", "DEV_QTR"]

selected_keys = (
    t0_position[key_columns]
    .drop_duplicates()
    .reset_index(drop=True)
)
selected_key_index = pd.MultiIndex.from_frame(selected_keys)

filtered_batches = []
rows_scanned = 0
rows_retained = 0
batch_size = 100_000

print(
    "Streaming modelling features from the master parquet in",
    f"{batch_size:,}-row batches..."
)

for batch_number, record_batch in enumerate(
    parquet_file.iter_batches(
        batch_size=batch_size,
        columns=feature_source_columns,
    ),
    start=1,
):
    chunk = record_batch.to_pandas()
    rows_scanned += len(chunk)

    chunk_key_index = pd.MultiIndex.from_frame(chunk[key_columns])
    keep_mask = chunk_key_index.isin(selected_key_index)

    if keep_mask.any():
        retained = chunk.loc[keep_mask].copy()
        filtered_batches.append(retained)
        rows_retained += len(retained)

    del chunk, chunk_key_index, keep_mask, record_batch

    if batch_number % 10 == 0:
        gc.collect()
        print(
            f"  scanned {rows_scanned:,} rows; "
            f"retained {rows_retained:,}"
        )

if not filtered_batches:
    raise ValueError(
        "No T0 modelling rows were recovered from the master parquet."
    )

deployment_features = pd.concat(
    filtered_batches,
    ignore_index=True,
)
del filtered_batches, selected_key_index
gc.collect()

duplicate_key_count = int(
    deployment_features.duplicated(key_columns).sum()
)
if duplicate_key_count:
    raise ValueError(
        f"{duplicate_key_count} duplicate feature keys were recovered."
    )

recovered_keys = deployment_features[key_columns].drop_duplicates()
missing_keys = (
    selected_keys
    .merge(
        recovered_keys,
        on=key_columns,
        how="left",
        indicator=True,
    )
    .loc[lambda x: x["_merge"].eq("left_only")]
)

if len(missing_keys):
    raise ValueError(
        f"{len(missing_keys)} selected T0 rows were not recovered "
        "from the master parquet."
    )

deployment_features["CUM_INC_LARGE"] = pd.to_numeric(
    deployment_features["CUM_INC_LARGE"], errors="coerce"
).fillna(0.0)
deployment_features["is_bi_excess_at_snapshot"] = (
    deployment_features["CUM_INC_LARGE"] > 0
).astype("int8")

for col in CATEGORICAL_FEATURES:
    deployment_features[col] = (
        deployment_features[col]
        .astype("object")
        .where(deployment_features[col].notna(), np.nan)
    )

for col in NUMERIC_FEATURES:
    if col == "is_bi_excess_at_snapshot":
        deployment_features[col] = (
            deployment_features[col].fillna(0).astype("int8")
        )
    else:
        deployment_features[col] = pd.to_numeric(
            deployment_features[col], errors="coerce"
        )

print(
    "Recovered deployment feature rows:",
    f"{len(deployment_features):,}",
)

## Deploy Random Forest and XGBoost

The models are applied only to the three supported maturity bands.
Cohorts over DEV_QTR 15 retain their full known incurred position at \(T_0\)
in this first deployment. This assumption is assessed separately through
their actual subsequent emergence.

In [ ]:
deployment = t0_position.merge(
    deployment_features[
        ["SOURCE_FILE", "CLAIMS_KEY", "DEV_QTR"] + FEATURE_COLUMNS
    ],
    on=["SOURCE_FILE", "CLAIMS_KEY", "DEV_QTR"],
    how="left",
    validate="one_to_one",
    suffixes=("", "_feature"),
)

del deployment_features
gc.collect()

deployment["rf_predicted_future"] = 0.0
deployment["xgb_predicted_future"] = 0.0

for snapshot in SNAPSHOT_MODELS:
    mask = (
        deployment["included_in_main_diagonal"]
        & deployment["model_band"].eq(snapshot)
    )

    X_band = deployment.loc[mask, FEATURE_COLUMNS].copy()

    if len(X_band) == 0:
        warnings.warn(f"No deployment rows found for {snapshot}.")
        continue

    print(f"Loading and deploying models for {snapshot}...")

    rf_model = joblib.load(RF_MODEL_PATHS[snapshot])
    rf_prediction = rf_model.predict(X_band)

    xgb_bundle = joblib.load(XGB_MODEL_PATHS[snapshot])
    xgb_preprocessor = xgb_bundle["preprocessor"]
    xgb_model = xgb_bundle["model"]
    X_band_processed = xgb_preprocessor.transform(X_band)
    xgb_prediction = xgb_model.predict(X_band_processed)

    deployment.loc[mask, "rf_predicted_future"] = rf_prediction
    deployment.loc[mask, "xgb_predicted_future"] = xgb_prediction

    print(
        snapshot,
        "rows=", len(X_band),
        "RF future=", f"{rf_prediction.sum():,.2f}",
        "XGB future=", f"{xgb_prediction.sum():,.2f}",
    )

    del (
        X_band,
        X_band_processed,
        rf_model,
        xgb_bundle,
        xgb_preprocessor,
        xgb_model,
        rf_prediction,
        xgb_prediction,
    )
    gc.collect()

deployment["rf_projected_latest_bixs"] = np.maximum(
    deployment["CUM_INC_LARGE"] + deployment["rf_predicted_future"],
    0.0,
)
deployment["xgb_projected_latest_bixs"] = np.maximum(
    deployment["CUM_INC_LARGE"] + deployment["xgb_predicted_future"],
    0.0,
)

main_deployment = deployment.loc[
    deployment["included_in_main_diagonal"]
    & ~deployment["model_band"].eq("UNSUPPORTED_EARLY")
].copy()

main_deployment.to_parquet(
    OUTPUT_FOLDER / "historical_diagonal_claim_level_predictions.parquet",
    index=False,
)

## Accident-year Random Forest and XGBoost results

This section gives a summary of the historical back-test results for each accident year using the two machine-learning models. For every accident year, it adds up the observed BI Excess at T0, the actual later amount at T1, and the RF/XGBoost projections. It then works out each model’s monetary error, absolute error, and percentage error.

In [ ]:
ml_ay = (
    main_deployment
    .groupby("ACC_YEAR", as_index=False)
    .agg(
        claim_count=("CLAIMS_KEY", "count"),
        incurred_bixs_T0=("CUM_INC_LARGE", "sum"),
        actual_future_development_T0_to_T1=(
            "actual_future_development_T0_to_T1", "sum"
        ),
        actual_latest_bixs_T1=("actual_latest_bixs_T1", "sum"),
        rf_predicted_future=("rf_predicted_future", "sum"),
        rf_projected_latest_bixs=("rf_projected_latest_bixs", "sum"),
        xgb_predicted_future=("xgb_predicted_future", "sum"),
        xgb_projected_latest_bixs=("xgb_projected_latest_bixs", "sum"),
    )
)

for method in ["rf", "xgb"]:
    ml_ay[f"{method}_error"] = (
        ml_ay[f"{method}_projected_latest_bixs"]
        - ml_ay["actual_latest_bixs_T1"]
    )
    ml_ay[f"{method}_absolute_error"] = ml_ay[
        f"{method}_error"
    ].abs()
    ml_ay[f"{method}_percentage_error"] = np.where(
        ml_ay["actual_latest_bixs_T1"] != 0,
        ml_ay[f"{method}_error"]
        / ml_ay["actual_latest_bixs_T1"],
        np.nan,
    )

display(ml_ay)

## Outcome scope reconciliation

Random Forest and XGBoost cover only claims reported by \(T_0\). Their
observed comparison is therefore the later incurred amount for those same
reported claims.

The incurred Chain Ladder is an aggregate total-portfolio method. Its observed
comparison includes both claims reported by \(T_0\) and claims first appearing
after \(T_0\). The latter amount is shown as the observed pure IBNR emergence.

A direct total-portfolio RF/XGBoost versus Chain Ladder comparison is deferred
until the macro pure IBNR allowance is added.

In [ ]:
eligible_t1_all_claims = t1_position.loc[
    t1_position["ACC_YEAR"] <= eligible_year_max
].copy()

full_actual_ay = (
    eligible_t1_all_claims
    .groupby("ACC_YEAR", as_index=False)
    .agg(
        full_portfolio_actual_latest_bixs_T1=("CUM_INC_LARGE", "sum"),
        full_portfolio_claim_count_T1=("CLAIMS_KEY", "count"),
    )
)

scope_reconciliation = ml_ay[
    [
        "ACC_YEAR",
        "actual_latest_bixs_T1",
        "claim_count",
    ]
].rename(
    columns={
        "actual_latest_bixs_T1": "reported_at_T0_actual_latest_bixs_T1",
        "claim_count": "reported_at_T0_claim_count",
    }
).merge(
    full_actual_ay,
    on="ACC_YEAR",
    how="outer",
)

for col in [
    "reported_at_T0_actual_latest_bixs_T1",
    "full_portfolio_actual_latest_bixs_T1",
    "reported_at_T0_claim_count",
    "full_portfolio_claim_count_T1",
]:
    scope_reconciliation[col] = scope_reconciliation[col].fillna(0.0)

scope_reconciliation["observed_pure_ibnr_bixs_T1"] = (
    scope_reconciliation["full_portfolio_actual_latest_bixs_T1"]
    - scope_reconciliation["reported_at_T0_actual_latest_bixs_T1"]
)
scope_reconciliation["observed_pure_ibnr_claim_count"] = (
    scope_reconciliation["full_portfolio_claim_count_T1"]
    - scope_reconciliation["reported_at_T0_claim_count"]
)

scope_reconciliation.to_csv(
    OUTPUT_FOLDER / "historical_diagonal_outcome_scope_reconciliation.csv",
    index=False,
)

display(scope_reconciliation)

## Incurred Chain Ladder projected to \(T_1\)

The Chain Ladder uses only cumulative incurred BI Excess cells observable
at \(T_0\). It projects each accident quarter from its maturity at \(T_0\)
to the maturity reached at \(T_1\), rather than beyond the available
comparison date.

In [ ]:
triangle_source = triangle_date_df.loc[
    triangle_date_df["VALUATION_QTR_INDEX"] <= T0_INDEX,
    [
        "ACCIDENT_QTR_INDEX",
        "ACC_YEAR",
        "DEV_QTR",
        "CUM_INC_LARGE",
    ],
].copy()

raw_triangle = (
    triangle_source
    .groupby(
        ["ACCIDENT_QTR_INDEX", "ACC_YEAR", "DEV_QTR"],
        as_index=False,
    )
    .agg(cumulative_incurred_bixs=("CUM_INC_LARGE", "sum"))
)

completed_rows = []

for origin, origin_df in raw_triangle.groupby("ACCIDENT_QTR_INDEX"):
    accident_year = int(origin_df["ACC_YEAR"].iloc[0])
    max_dev = int(T0_INDEX - origin)

    if max_dev < 0:
        continue

    origin_series = (
        origin_df
        .set_index("DEV_QTR")["cumulative_incurred_bixs"]
        .sort_index()
    )

    full_index = pd.Index(range(0, max_dev + 1), name="DEV_QTR")
    origin_series = (
        origin_series
        .reindex(full_index)
        .ffill()
        .fillna(0.0)
    )

    completed_rows.append(
        pd.DataFrame({
            "ACCIDENT_QTR_INDEX": origin,
            "ACC_YEAR": accident_year,
            "DEV_QTR": full_index,
            "cumulative_incurred_bixs": origin_series.to_numpy(),
        })
    )

completed_triangle = pd.concat(completed_rows, ignore_index=True)

max_development = int(completed_triangle["DEV_QTR"].max())
factor_rows = []

for dev in range(max_development):
    current_cells = completed_triangle.loc[
        completed_triangle["DEV_QTR"].eq(dev),
        ["ACCIDENT_QTR_INDEX", "cumulative_incurred_bixs"],
    ].rename(
        columns={"cumulative_incurred_bixs": "current"}
    )

    next_cells = completed_triangle.loc[
        completed_triangle["DEV_QTR"].eq(dev + 1),
        ["ACCIDENT_QTR_INDEX", "cumulative_incurred_bixs"],
    ].rename(
        columns={"cumulative_incurred_bixs": "next"}
    )

    paired = current_cells.merge(
        next_cells,
        on="ACCIDENT_QTR_INDEX",
        how="inner",
    )

    contributing = paired.loc[paired["current"] > 0].copy()
    origin_count = len(contributing)
    denominator = float(contributing["current"].sum())
    numerator = float(contributing["next"].sum())

    if origin_count >= MIN_ORIGINS_PER_CL_FACTOR and denominator > 0:
        factor = numerator / denominator
        usable = True
    else:
        factor = 1.0
        usable = False

    factor_rows.append({
        "development_quarter": dev,
        "origin_count": origin_count,
        "denominator": denominator,
        "numerator": numerator,
        "factor": factor,
        "factor_usable": usable,
    })

cl_factors = pd.DataFrame(factor_rows)
cl_factors.to_csv(
    OUTPUT_FOLDER / "historical_incurred_chain_ladder_factors.csv",
    index=False,
)

display(cl_factors.head(20))

In [ ]:
origin_latest_t0 = (
    completed_triangle
    .sort_values(["ACCIDENT_QTR_INDEX", "DEV_QTR"])
    .drop_duplicates("ACCIDENT_QTR_INDEX", keep="last")
    .rename(
        columns={
            "DEV_QTR": "development_at_T0",
            "cumulative_incurred_bixs": "incurred_bixs_T0",
        }
    )
)

factor_lookup = cl_factors.set_index("development_quarter")["factor"].to_dict()
usable_lookup = (
    cl_factors.set_index("development_quarter")["factor_usable"].to_dict()
)

cl_projection_rows = []

for row in origin_latest_t0.itertuples(index=False):
    current_dev = int(row.development_at_T0)
    target_dev = int(T1_INDEX - row.ACCIDENT_QTR_INDEX)
    projected = float(row.incurred_bixs_T0)
    missing_factor_count = 0

    for dev in range(current_dev, max(target_dev, current_dev)):
        factor = factor_lookup.get(dev, 1.0)
        factor_usable = usable_lookup.get(dev, False)

        if not factor_usable:
            missing_factor_count += 1

        projected *= factor

    cl_projection_rows.append({
        "ACCIDENT_QTR_INDEX": row.ACCIDENT_QTR_INDEX,
        "ACC_YEAR": row.ACC_YEAR,
        "development_at_T0": current_dev,
        "target_development_at_T1": target_dev,
        "incurred_bixs_T0": float(row.incurred_bixs_T0),
        "cl_projected_latest_bixs_T1": projected,
        "missing_or_unusable_factor_count": missing_factor_count,
    })

cl_origin_projection = pd.DataFrame(cl_projection_rows)

if EXCLUDE_CURRENT_ACCIDENT_YEAR:
    cl_origin_projection = cl_origin_projection.loc[
        cl_origin_projection["ACC_YEAR"] <= eligible_year_max
    ].copy()

cl_ay = (
    cl_origin_projection
    .groupby("ACC_YEAR", as_index=False)
    .agg(
        cl_incurred_bixs_T0=("incurred_bixs_T0", "sum"),
        cl_projected_latest_bixs_T1=(
            "cl_projected_latest_bixs_T1", "sum"
        ),
        cl_missing_factor_count=(
            "missing_or_unusable_factor_count", "sum"
        ),
    )
)

cl_origin_projection.to_csv(
    OUTPUT_FOLDER / "historical_cl_projection_by_accident_quarter.csv",
    index=False,
)

display(cl_ay)

## Final accident-year and portfolio comparison

In this section, we build the final accident-year comparison table by bringing together the machine learning results, the reconciliation of reported and pure IBNR claims, and the Chain Ladder results. Next, we calculate the errors for RF and XGBoost using the reported-claim scope, compare Chain Ladder against the full portfolio scope, and save the combined table.

In [ ]:
final_ay = (
    ml_ay
    .merge(
        scope_reconciliation[
            [
                "ACC_YEAR",
                "reported_at_T0_actual_latest_bixs_T1",
                "full_portfolio_actual_latest_bixs_T1",
                "observed_pure_ibnr_bixs_T1",
            ]
        ],
        on="ACC_YEAR",
        how="left",
        validate="one_to_one",
    )
    .merge(
        cl_ay,
        on="ACC_YEAR",
        how="left",
        validate="one_to_one",
    )
)

# RF and XGBoost errors: reported-claim scope.
for method in ["rf", "xgb"]:
    final_ay[f"{method}_reported_error"] = (
        final_ay[f"{method}_projected_latest_bixs"]
        - final_ay["reported_at_T0_actual_latest_bixs_T1"]
    )
    final_ay[f"{method}_reported_absolute_error"] = final_ay[
        f"{method}_reported_error"
    ].abs()
    final_ay[f"{method}_reported_percentage_error"] = np.where(
        final_ay["reported_at_T0_actual_latest_bixs_T1"] != 0,
        final_ay[f"{method}_reported_error"]
        / final_ay["reported_at_T0_actual_latest_bixs_T1"],
        np.nan,
    )

# Chain Ladder error: total-portfolio scope.
final_ay["cl_total_error"] = (
    final_ay["cl_projected_latest_bixs_T1"]
    - final_ay["full_portfolio_actual_latest_bixs_T1"]
)
final_ay["cl_total_absolute_error"] = (
    final_ay["cl_total_error"].abs()
)
final_ay["cl_total_percentage_error"] = np.where(
    final_ay["full_portfolio_actual_latest_bixs_T1"] != 0,
    final_ay["cl_total_error"]
    / final_ay["full_portfolio_actual_latest_bixs_T1"],
    np.nan,
)

final_ay.to_csv(
    OUTPUT_FOLDER / "historical_diagonal_comparison_by_accident_year.csv",
    index=False,
)

display(final_ay)

This section creates the portfolio-level summary for the historical back-test. It totals the observed and projected BI Excess amounts for each method, calculates overall error, bias ratio, and accident-year WAPE, and records valuation dates and accident-year range.

The warning at the end is important: RF and XGBoost are still assessed only on claims reported at T0T_0, whereas Chain Ladder includes the whole portfolio, including pure IBNR. So this table is not yet a fully like-for-like ranking until the separate pure IBNR allowance is added to the machine-learning projections.

In [ ]:
def method_summary(
    method_name,
    scope,
    observed_col,
    projected_col,
    absolute_error_col,
):
    observed_total = float(final_ay[observed_col].sum())
    projected_total = float(final_ay[projected_col].sum())
    total_error = projected_total - observed_total
    ay_wape = (
        float(final_ay[absolute_error_col].sum()) / abs(observed_total)
        if observed_total != 0
        else np.nan
    )

    return {
        "method": method_name,
        "scope": scope,
        "T0": T0_LABEL,
        "T1": T1_LABEL,
        "accident_year_min": int(final_ay["ACC_YEAR"].min()),
        "accident_year_max": int(final_ay["ACC_YEAR"].max()),
        "incurred_bixs_T0": float(final_ay["incurred_bixs_T0"].sum()),
        "observed_latest_bixs_T1": observed_total,
        "projected_latest_bixs_T1": projected_total,
        "difference_projected_minus_observed": total_error,
        "bias_ratio": (
            projected_total / observed_total
            if observed_total != 0
            else np.nan
        ),
        "accident_year_wape": ay_wape,
    }


portfolio_summary = pd.DataFrame([
    method_summary(
        "Random Forest",
        "Reported claims visible at T0",
        "reported_at_T0_actual_latest_bixs_T1",
        "rf_projected_latest_bixs",
        "rf_reported_absolute_error",
    ),
    method_summary(
        "XGBoost",
        "Reported claims visible at T0",
        "reported_at_T0_actual_latest_bixs_T1",
        "xgb_projected_latest_bixs",
        "xgb_reported_absolute_error",
    ),
    method_summary(
        "Incurred Chain Ladder",
        "Total portfolio, including pure IBNR",
        "full_portfolio_actual_latest_bixs_T1",
        "cl_projected_latest_bixs_T1",
        "cl_total_absolute_error",
    ),
])

portfolio_summary.to_csv(
    OUTPUT_FOLDER / "historical_diagonal_portfolio_summary.csv",
    index=False,
)

display(portfolio_summary)

print(
    "Do not rank RF/XGBoost directly against Chain Ladder from this "
    "table until the pure IBNR allowance has been added to the ML "
    "reported-claim projections."
)

## Mature-cohort and unsupported-cohort checks

The first deployment assumes no additional ML tail after DEV_QTR 15.
The following table shows whether that assumption is material when compared
with actual subsequent emergence to \(T_1\).

In [ ]:
assumption_check = (
    deployment
    .groupby(
        ["included_in_main_diagonal", "model_band"],
        as_index=False,
    )
    .agg(
        claim_count=("CLAIMS_KEY", "count"),
        incurred_bixs_T0=("CUM_INC_LARGE", "sum"),
        observed_latest_bixs_T1=("actual_latest_bixs_T1", "sum"),
        observed_future_development=(
            "actual_future_development_T0_to_T1", "sum"
        ),
    )
)

assumption_check["future_share_of_T1"] = np.where(
    assumption_check["observed_latest_bixs_T1"] != 0,
    assumption_check["observed_future_development"]
    / assumption_check["observed_latest_bixs_T1"],
    np.nan,
)

assumption_check.to_csv(
    OUTPUT_FOLDER / "historical_diagonal_assumption_check.csv",
    index=False,
)

display(assumption_check)

mature_row = assumption_check.loc[
    assumption_check["included_in_main_diagonal"]
    & assumption_check["model_band"].eq("MATURE_POSITION")
]

if not mature_row.empty:
    mature_future_share = float(
        mature_row["future_share_of_T1"].iloc[0]
    )
    if abs(mature_future_share) > 0.05:
        warnings.warn(
            "Observed post-T0 development for the mature-position group "
            "exceeds 5% of its T1 amount. A separate tail treatment should "
            "be developed before treating this as the final diagonal."
        )